# Collectors

*Run a cell with ▶ (or Shift+Enter): Sphinx compiles and runs it on your computer. Change the code and run it again: this notebook is your own copy.*

**After this unit you can:**
- Collect a stream into a list, a set, a String or a map with collect and Collectors.
- Group and split with groupingBy and partitioningBy, and flatten nested lists with flatMap.

**In this notebook**, one idea at a time (run each cell before moving on):
1. collect: lists, sets and Strings
2. toMap: one value per key
3. Grouping with collectors
4. Grouping and then counting or mapping
5. partitioningBy: two groups
6. flatMap: flattening nested lists

**Exercises** in this notebook end with `check(...)` lines: minimal tests that print ✓ when your code is right. Run the next cell once first, so they can use `check`.

In [ ]:
// shared: check(...) prints ✓ or ✗; the exercises below use it
void check(String what, Object actual, Object expected) {
    if (Objects.equals(actual, expected)) {
        IO.println("✓ " + what);
    } else {
        IO.println("✗ " + what + ": expected " + expected + ", got " + actual);
    }
}

## 1. collect: lists, sets and Strings

`collect(...)` is the terminal operation that builds a container from the stream's items. `Collectors.toSet()` drops repeats, `Collectors.toCollection(TreeSet::new)` picks the kind of collection, and `Collectors.joining(", ", "[", "]")` glues Strings with a separator (and an optional start and end).

In [ ]:
List<String> tags = List.of("java", "c", "java", "sql", "c");
Set<String> unique = tags.stream().collect(Collectors.toCollection(TreeSet::new));
IO.println(unique);
String line = tags.stream().distinct().collect(Collectors.joining(" | ", "<", ">"));
IO.println(line);

🧪 **Try it:** join the tags in upper case, separated by commas, with no start or end.

### ✏️ Exercise: Numbers as one line

Return the numbers joined by `;`, such as `"1;2;3"`. Turn each into a String with `map`, then `Collectors.joining`.

In [ ]:
String joined(List<Integer> values) {
    // TODO: map(String::valueOf) then collect(joining(";"))
    return "";
}

void main() {
    check("three numbers", joined(List.of(1, 2, 3)), "1;2;3");
    check("one", joined(List.of(42)), "42");
    check("none", joined(List.of()), "");
}

<details>
<summary>Show an answer</summary>

```java
String joined(List<Integer> values) {
    return values.stream().map(String::valueOf).collect(Collectors.joining(";"));
}

void main() {
    check("three numbers", joined(List.of(1, 2, 3)), "1;2;3");
    check("one", joined(List.of(42)), "42");
    check("none", joined(List.of()), "");
}
```

</details>

## 2. toMap: one value per key

`Collectors.toMap(keyFunction, valueFunction)` builds a map with one entry per item. Two items with the same key make it throw an `IllegalStateException`, unless a third argument says how to merge their values.

In [ ]:
record Product(String name, double price) {}

void main() {
    List<Product> items = List.of(new Product("pen", 2.5), new Product("book", 30), new Product("pen", 3));
    try {
        items.stream().collect(Collectors.toMap(Product::name, Product::price));
    } catch (IllegalStateException e) {
        IO.println("Duplicate key: " + e.getMessage());
    }
    Map<String, Double> cheapest = items.stream()
            .collect(Collectors.toMap(Product::name, Product::price, Math::min));
    IO.println(cheapest);
}

🧪 **Try it:** change the merge function from `Math::min` to `Double::sum`. What does each value mean now?

## 3. Grouping with collectors

`collect(Collectors.groupingBy(key))` builds a **Map** in one line: the key is computed for each item, and each key gets the list of its items. Pass a second collector to get something else per key, such as `Collectors.counting()`.

In [ ]:
List<String> animals = List.of("cat", "cow", "dog", "duck", "crow", "bee");
Map<Character, List<String>> byLetter = animals.stream()
        .collect(Collectors.groupingBy(a -> a.charAt(0)));
IO.println(byLetter);
Map<Character, Long> howMany = animals.stream()
        .collect(Collectors.groupingBy(a -> a.charAt(0), Collectors.counting()));
IO.println(howMany);

🧪 **Try it:** group by the **last** letter instead. Then compare with the loop you would need to build the same map by hand.

## 4. Grouping and then counting or mapping

`groupingBy(key)` alone gives lists of whole items. A **second collector** changes what each group becomes: `counting()` for its size, `mapping(f, toList())` for one field of each item, `summingInt(f)` or `averagingDouble(f)` for a total or an average. The slides follow a grouping.

#### 🎞️ Grouping, step by step

![groupingBy takes each city, computes its first letter as the key, and puts cities with the same key in the same list; with counting(), each group becomes its size](grouping-steps.svg)

In [ ]:
record Sale(String city, String product, int units) {}

void main() {
    List<Sale> sales = List.of(new Sale("Lima", "tea", 3), new Sale("Rome", "tea", 5),
            new Sale("Lima", "cake", 2), new Sale("Lima", "tea", 1));
    IO.println(sales.stream().collect(Collectors.groupingBy(Sale::city, TreeMap::new, Collectors.counting())));
    IO.println(sales.stream().collect(Collectors.groupingBy(Sale::city, TreeMap::new, Collectors.summingInt(Sale::units))));
    IO.println(sales.stream().collect(Collectors.groupingBy(Sale::city, TreeMap::new,
            Collectors.mapping(Sale::product, Collectors.toSet()))));
}

🧪 **Try it:** group by product instead of city, and sum the units.

## 5. partitioningBy: two groups

`partitioningBy(test)` always gives a map with exactly two keys, `true` and `false`: the items that pass the test and the ones that don't. Like `groupingBy`, it takes a second collector.

In [ ]:
List<Integer> temps = List.of(31, 18, 25, 35, 12);
Map<Boolean, List<Integer>> hot = temps.stream().collect(Collectors.partitioningBy(t -> t >= 30));
IO.println("hot: " + hot.get(true) + ", mild: " + hot.get(false));
IO.println(temps.stream().collect(Collectors.partitioningBy(t -> t >= 30, Collectors.counting())));

🧪 **Try it:** partition with a test nothing passes (`t > 100`): both keys are still there.

### ✏️ Exercise: Even and odd

Split the numbers into even (`true`) and odd (`false`) with `partitioningBy`.

In [ ]:
Map<Boolean, List<Integer>> evenOdd(List<Integer> values) {
    // TODO: collect(Collectors.partitioningBy(v -> v % 2 == 0))
    return Map.of();
}

void main() {
    check("mixed", evenOdd(List.of(1, 2, 3, 4)), Map.of(true, List.of(2, 4), false, List.of(1, 3)));
    check("empty", evenOdd(List.of()), Map.of(true, List.of(), false, List.of()));
}

<details>
<summary>Show an answer</summary>

```java
Map<Boolean, List<Integer>> evenOdd(List<Integer> values) {
    return values.stream().collect(Collectors.partitioningBy(v -> v % 2 == 0));
}

void main() {
    check("mixed", evenOdd(List.of(1, 2, 3, 4)), Map.of(true, List.of(2, 4), false, List.of(1, 3)));
    check("empty", evenOdd(List.of()), Map.of(true, List.of(), false, List.of()));
}
```

</details>

## 6. flatMap: flattening nested lists

`map` turns each item into exactly one item. `flatMap` turns each item into a **stream** of items and joins them all into one stream: lines into words, orders into their products, a list of lists into one list.

In [ ]:
List<List<String>> orders = List.of(List.of("tea", "cake"), List.of("tea"), List.of("juice", "cake"));
List<String> all = orders.stream().flatMap(List::stream).toList();
IO.println(all);
IO.println(all.stream().distinct().sorted().toList());
List<String> lines = List.of("to be", "or not");
IO.println(lines.stream().flatMap(l -> Arrays.stream(l.split(" "))).toList());

🧪 **Try it:** replace `flatMap(List::stream)` with `map(List::stream)` and print it: you get a list of streams, not of products.

## In short

`collect(...)` is the terminal operation that builds a container from a stream, and the **Collectors** class makes the common ones: `joining`, `toSet`, `toMap`, `groupingBy` and `partitioningBy`.

```java
void main() {
    List<String> cities = List.of("Lima", "Lisbon", "Paris", "Porto", "Rome");
    IO.println(cities.stream().collect(Collectors.joining(", ")));
    Map<Character, List<String>> byLetter = cities.stream()
            .collect(Collectors.groupingBy(c -> c.charAt(0)));
    IO.println(byLetter);          // {P=[Paris, Porto], R=[Rome], L=[Lima, Lisbon]}
    Map<Boolean, Long> short4 = cities.stream()
            .collect(Collectors.partitioningBy(c -> c.length() <= 4, Collectors.counting()));
    IO.println(short4);            // {false=3, true=2}

List<List<Integer>> rows = List.of(List.of(1, 2), List.of(3));
    IO.println(rows.stream().flatMap(List::stream).toList());   // [1, 2, 3]
}
```

`groupingBy` gives a map from each key to its items; a second collector, such as `counting()`, turns each group into something else.

**Watch out:** `toMap` throws an exception when two items have the same key, unless you say how to merge them.

## Readings

- [Using a Collector as a Terminal Operation](https://dev.java/learn/api/collections-and-streams/streams/using-collectors/) (dev.java (Oracle), 20 min): The sections "Collecting in a String of Characters", "Partitioning Elements with a Predicate" and "Collecting in a Map with Grouping By". You can stop before the post-processing sections.

## Practice

The notebook covered the basics. Now apply them in the unit's challenges, in the Sphinx sidebar, where Submit runs every test, hidden ones included:

- **Group Words by Length** · Medium
- **Pass or Fail (partitioningBy)** · Medium
- **Word Index (flatMap)** · Hard
- Test: **Student Report** · Hard